# Results Analysis - Crypto Price Prediction

This notebook consolidates all model evaluation results, compares performance
across models, analyses market-state impact, anomaly correlations, and
attention patterns.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

# Ensure the project root is importable
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

from src.utils.config import load_config
from src.utils.io import load_metrics, load_dataframe
from src.utils.constants import *
from src.analysis.market_state import MarketStateClassifier
from src.analysis.anomaly_analysis import AnomalyAnalyzer
from src.analysis.attention_viz import AttentionVisualizer

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

cfg = load_config()
print("Config loaded.")

## 1. Load All Metrics

In [ ]:
# Discover all metrics JSON files in results/metrics/
metrics_dir = METRICS_DIR
metric_files = sorted(metrics_dir.glob("*.json"))

print(f"Found {len(metric_files)} metric files in {metrics_dir}:")
for f in metric_files:
    print(f"  - {f.name}")

# Load into a dict keyed by filename stem
all_metrics: dict[str, dict] = {}
for f in metric_files:
    all_metrics[f.stem] = load_metrics(f)

print(f"\nLoaded metrics for {len(all_metrics)} experiments.")

## 2. Build Comparison Table

In [ ]:
# Build a DataFrame where each row is a model/experiment and columns are metrics
rows = []
for name, m in all_metrics.items():
    row = {"experiment": name}
    # Flatten nested dicts (e.g. {"reg_mae": 0.1} or {"mae": 0.1})
    for key, val in m.items():
        if isinstance(val, dict):
            for sub_key, sub_val in val.items():
                row[f"{key}_{sub_key}"] = sub_val
        else:
            row[key] = val
    rows.append(row)

comparison_df = pd.DataFrame(rows).set_index("experiment")
print(f"Comparison table: {comparison_df.shape}")
comparison_df

## 3. Best Model Per Metric

In [ ]:
# For each numeric column, identify the best model
# Lower is better: mae, rmse, mape
# Higher is better: directional_accuracy, accuracy, f1_score, auc_roc
LOWER_IS_BETTER = {"mae", "rmse", "mape", "reg_mae", "reg_rmse", "reg_mape"}
HIGHER_IS_BETTER = {
    "directional_accuracy", "reg_directional_accuracy",
    "accuracy", "cls_accuracy",
    "f1_score", "cls_f1_score",
    "auc_roc", "cls_auc_roc",
}

numeric_cols = comparison_df.select_dtypes(include=[np.number]).columns
best_models = []

for col in numeric_cols:
    series = comparison_df[col].dropna()
    if series.empty:
        continue
    col_lower = col.lower()
    if any(m in col_lower for m in LOWER_IS_BETTER):
        best_idx = series.idxmin()
        best_val = series.min()
        direction = "min"
    elif any(m in col_lower for m in HIGHER_IS_BETTER):
        best_idx = series.idxmax()
        best_val = series.max()
        direction = "max"
    else:
        best_idx = series.idxmin()
        best_val = series.min()
        direction = "min (default)"
    best_models.append({"metric": col, "best_model": best_idx, "value": best_val, "direction": direction})

best_df = pd.DataFrame(best_models)
print("Best model per metric:")
best_df

## 4. Visualisation: Bar Charts Comparing Models

In [ ]:
# Select key regression metrics for comparison
reg_metric_cols = [c for c in numeric_cols if any(m in c.lower() for m in ("mae", "rmse", "mape"))]

if reg_metric_cols:
    fig = make_subplots(
        rows=1,
        cols=len(reg_metric_cols),
        subplot_titles=[c.upper() for c in reg_metric_cols],
        shared_yaxes=False,
    )
    colours = px.colors.qualitative.Set2

    for i, col in enumerate(reg_metric_cols, 1):
        series = comparison_df[col].dropna().sort_values()
        fig.add_trace(
            go.Bar(
                x=series.index.tolist(),
                y=series.values.tolist(),
                marker_color=colours[:len(series)],
                name=col,
                showlegend=False,
            ),
            row=1,
            col=i,
        )

    fig.update_layout(height=450, title_text="Regression Metrics Comparison")
    fig.show()
else:
    print("No regression metric columns found.")

In [ ]:
# Classification metrics comparison
cls_metric_cols = [c for c in numeric_cols if any(m in c.lower() for m in ("accuracy", "f1", "auc"))]

if cls_metric_cols:
    fig = make_subplots(
        rows=1,
        cols=len(cls_metric_cols),
        subplot_titles=[c.upper() for c in cls_metric_cols],
        shared_yaxes=False,
    )
    colours = px.colors.qualitative.Pastel

    for i, col in enumerate(cls_metric_cols, 1):
        series = comparison_df[col].dropna().sort_values(ascending=False)
        fig.add_trace(
            go.Bar(
                x=series.index.tolist(),
                y=series.values.tolist(),
                marker_color=colours[:len(series)],
                name=col,
                showlegend=False,
            ),
            row=1,
            col=i,
        )

    fig.update_layout(height=450, title_text="Classification Metrics Comparison")
    fig.show()
else:
    print("No classification metric columns found.")

## 5. Market State Analysis

In [ ]:
# Load processed price data for market-state classification
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "BTC_USDT_1h.parquet"

try:
    df_price = load_dataframe(DATA_PATH)
    print(f"Price data loaded: {df_price.shape}")
    print(f"Date range: {df_price.index.min()} -> {df_price.index.max()}")
except FileNotFoundError:
    df_price = None
    print(f"Price data not found at {DATA_PATH}. Skipping market-state analysis.")

In [ ]:
if df_price is not None:
    # Classify market states
    classifier = MarketStateClassifier(
        ema_short_period=10,
        ema_long_period=50,
        return_threshold=0.02,
    )
    states = classifier.classify(df_price, window=20)

    # Distribution
    dist = classifier.get_state_distribution(states)
    dist_df = pd.DataFrame(dist).T
    print("Market State Distribution:")
    display(dist_df)

    # Plot
    fig = classifier.plot_states(df_price, states)
    fig.show()
else:
    print("Skipped — no price data available.")

In [ ]:
# Per-state metrics (requires prediction arrays — placeholder)
# Replace y_true / y_pred with actual test-set predictions when available.
if df_price is not None:
    print("To compute per-state metrics, load y_true and y_pred from your test results:")
    print("  metrics_by_state = classifier.compute_metrics_by_state(y_true, y_pred, states)")
    print("  pd.DataFrame(metrics_by_state).T")

## 6. Anomaly Impact Analysis

In [ ]:
# Anomaly analysis requires:
#   - anomaly_flags: boolean array from the anomaly detector
#   - prediction_errors: absolute prediction errors
#   - timestamps: datetime index
#
# Example (uncomment and adapt when data is available):
#
# anomaly_flags = np.load(PROJECT_ROOT / "results" / "anomaly_flags.npy")
# prediction_errors = np.load(PROJECT_ROOT / "results" / "prediction_errors.npy")
# timestamps = df_price.index[:len(anomaly_flags)]
#
# analyzer = AnomalyAnalyzer(anomaly_flags, prediction_errors, timestamps)
# report = analyzer.generate_report()

print("Anomaly analysis template ready.")
print("Load anomaly_flags, prediction_errors, and timestamps to run the full analysis.")
print()
print("Example usage:")
print("  analyzer = AnomalyAnalyzer(anomaly_flags, prediction_errors, timestamps)")
print("  report = analyzer.generate_report()")
print("  print(json.dumps(report['summary'], indent=2))")

In [ ]:
# Known-event detection check (can run even without model predictions)
if df_price is not None:
    print("Known crypto events that fall within the data range:")
    # Create dummy anomaly flags to check event coverage
    dummy_flags = np.zeros(len(df_price), dtype=bool)
    events = AnomalyAnalyzer.identify_known_events(df_price.index, dummy_flags)
    if events:
        events_df = pd.DataFrame(events)
        display(events_df[["event_name", "start", "end", "datapoints_in_window"]])
    else:
        print("  No known events overlap with the data range.")

## 7. Attention Visualization

In [ ]:
# Attention visualisation requires a trained Transformer or TFT model.
#
# Example (uncomment and adapt):
#
# from src.models.transformer_model import TransformerModel  # or TFTModel
# from src.utils.io import load_model
#
# model = TransformerModel(...)  # initialise with the right architecture params
# model.load(CHECKPOINTS_DIR / "transformer_BTC_USDT_1h.pt")
#
# viz = AttentionVisualizer(model, feature_names=FEATURE_COLUMNS)
#
# # Feature importance
# importance_df = viz.get_feature_importance(X_test)
# display(importance_df.head(10))
# viz.plot_feature_importance(X_test)
#
# # Temporal attention
# viz.plot_temporal_attention(X_test, sample_idx=0)
#
# # Attention over time
# viz.plot_attention_over_time(X_test, timestamps=test_timestamps)

print("Attention visualization template ready.")
print("Load a trained Transformer/TFT model and test data to generate visualizations.")

## 8. Statistical Significance Tests Between Models

In [ ]:
def paired_significance_test(
    errors_a: np.ndarray,
    errors_b: np.ndarray,
    model_a_name: str,
    model_b_name: str,
    alpha: float = 0.05,
) -> dict:
    """Run a paired Wilcoxon signed-rank test on absolute prediction errors.

    Also runs the Diebold-Mariano-like comparison using a paired t-test
    on the loss differential.
    """
    errors_a = np.asarray(errors_a).ravel()
    errors_b = np.asarray(errors_b).ravel()
    assert len(errors_a) == len(errors_b), "Error arrays must have the same length."

    # Wilcoxon signed-rank test
    try:
        w_stat, w_pval = stats.wilcoxon(errors_a, errors_b)
    except ValueError:
        w_stat, w_pval = float("nan"), float("nan")

    # Paired t-test on loss differential (Diebold-Mariano proxy)
    loss_diff = errors_a ** 2 - errors_b ** 2
    t_stat, t_pval = stats.ttest_1samp(loss_diff, 0)

    return {
        "model_a": model_a_name,
        "model_b": model_b_name,
        "mean_error_a": float(np.mean(errors_a)),
        "mean_error_b": float(np.mean(errors_b)),
        "wilcoxon_statistic": float(w_stat),
        "wilcoxon_p_value": float(w_pval),
        "wilcoxon_significant": w_pval < alpha,
        "dm_t_statistic": float(t_stat),
        "dm_p_value": float(t_pval),
        "dm_significant": t_pval < alpha,
        "winner": model_a_name if np.mean(errors_a) < np.mean(errors_b) else model_b_name,
    }


print("Function `paired_significance_test` defined.")
print()
print("Usage:")
print("  result = paired_significance_test(errors_lstm, errors_tft, 'LSTM', 'TFT')")
print("  print(json.dumps(result, indent=2))")

In [ ]:
# Pairwise significance testing across all models
# (requires per-sample prediction errors for each model)
#
# Example:
# model_errors = {
#     "LSTM": np.abs(y_true - y_pred_lstm),
#     "GRU": np.abs(y_true - y_pred_gru),
#     "Transformer": np.abs(y_true - y_pred_transformer),
#     "TFT": np.abs(y_true - y_pred_tft),
#     "XGBoost": np.abs(y_true - y_pred_xgboost),
# }
#
# sig_results = []
# model_names = list(model_errors.keys())
# for i in range(len(model_names)):
#     for j in range(i + 1, len(model_names)):
#         a, b = model_names[i], model_names[j]
#         result = paired_significance_test(model_errors[a], model_errors[b], a, b)
#         sig_results.append(result)
#
# sig_df = pd.DataFrame(sig_results)
# display(sig_df)
#
# # Summary: which model wins most comparisons?
# win_counts = sig_df["winner"].value_counts()
# print("\nWin counts (pairwise comparisons):")
# print(win_counts)

print("Pairwise significance test template ready.")
print("Load per-sample prediction errors for each model to run the full analysis.")

## 9. Summary

In [ ]:
print("=" * 60)
print("RESULTS ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nMetric files loaded: {len(all_metrics)}")
if not comparison_df.empty:
    print(f"Models evaluated: {list(comparison_df.index)}")
    print(f"Metrics tracked: {list(numeric_cols)}")
    print(f"\nBest overall models:")
    for _, row in best_df.iterrows():
        print(f"  {row['metric']:30s} -> {row['best_model']} ({row['value']:.6f})")
else:
    print("No metric data available yet. Train models first.")

print("\n" + "=" * 60)
print("Next steps:")
print("  1. Train all models and re-run this notebook")
print("  2. Load per-sample errors for significance tests")
print("  3. Load anomaly detector output for impact analysis")
print("  4. Load attention-based model for interpretation")
print("=" * 60)